# Entrenamiento de de dataset sintetico Random Forest

## SafeDrive — Pipeline completo

Ejecuta las secciones **en orden**:
1. Genera dataset sintético
2. Monta el dataset final, construir archivos train, test, validation
3. Entrena y exporta `.keras` + `.tflite`

In [ ]:
# Dependencias comunes a todo el notebook
import os
import glob
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split

print(f'TensorFlow {tf.__version__}')

---
## 1. Genera dataset sintético
Genera datos físicamente realistas para las 3 clases.

**Salida:** `ml/data/processed/source/datos_sinteticos.csv`

In [ ]:
for folder in ['ml/data/processed/source', 'ml/data/processed/dataset/wrong_data', 'models']:
    os.makedirs(folder, exist_ok=True)

def generar_datos(n_samples, label):
    if label == 0:   # Falso Positivo — conducción normal
        peak_g    = np.random.uniform(0.1,  1.5, n_samples)
        jerk      = np.random.uniform(0.1,  0.8, n_samples)
        audio     = np.random.uniform(0.0,  0.2, n_samples)
        angular   = np.random.uniform(0.0,  0.3, n_samples)
        velocidad = np.random.uniform(10,  120,  n_samples)
    elif label == 1: # Posible Susto — frenazo / derrape
        peak_g    = np.random.uniform(1.5,  4.0, n_samples)
        jerk      = np.random.uniform(0.8,  2.5, n_samples)
        audio     = np.random.uniform(0.1,  0.4, n_samples)
        angular   = np.random.uniform(0.3,  0.8, n_samples)
        velocidad = np.random.uniform(20,  100,  n_samples)
    elif label == 2: # Choque Grave
        peak_g    = np.random.uniform(4.0, 12.0, n_samples)
        jerk      = np.random.uniform(2.5,  8.0, n_samples)
        audio     = np.random.uniform(0.6,  1.0, n_samples)
        angular   = np.random.uniform(0.5,  2.0, n_samples)
        velocidad = np.random.uniform(30,  120,  n_samples)

    return pd.DataFrame({
        'Peak_G':         peak_g,
        'Jerk':           jerk,
        'Firma_Acustica': audio,
        'Cambio_Angular': angular,
        'Velocidad':      velocidad,
        'Label':          [label] * n_samples
    })

df_sintetico = pd.concat([
    generar_datos(4000, 0),
    generar_datos(1500, 1),
    generar_datos(1500, 2)
]).sample(frac=1, random_state=42).reset_index(drop=True)

# Dataset basura (outliers físicamente imposibles)
wrong_data = pd.DataFrame({
    'Peak_G':         [500.0, -2.0,   0.0,  999.9],
    'Jerk':           [0.0,  900.0, -10.0,    0.0],
    'Firma_Acustica': [2.5,   -0.5,   0.0,   10.0],
    'Cambio_Angular': [0.0,    0.0,   0.0,   50.0],
    'Velocidad':      [-50,    400,     0,      0],
    'Label':          [0,       1,     9,      2]
})

df_sintetico.to_csv('ml/data/processed/source/datos_sinteticos.csv', index=False)
wrong_data.to_csv('ml/data/processed/dataset/wrong_data/corrupted_data.csv', index=False)

print(f'Sintético: {len(df_sintetico)} filas')
print(df_sintetico['Label'].value_counts().sort_index())

---
## 2. Monta el dataset final
Fusiona todas las fuentes disponibles en `ml/data/processed/source/`, mezcla y divide en **train 70% / validation 15% / test 15%**.

In [ ]:
for folder in ['ml/data/processed/dataset/train',
               'ml/data/processed/dataset/validation',
               'ml/data/processed/dataset/test']:
    os.makedirs(folder, exist_ok=True)

# Leer todas las fuentes (wrong_data queda excluido por estar en otra carpeta)
archivos = glob.glob('ml/data/processed/source/*.csv')
print(f'Fuentes encontradas ({len(archivos)}):')
for f in archivos:
    df_tmp = pd.read_csv(f)
    print(f'  {f}  ->  {len(df_tmp)} filas')

dataset_completo = pd.concat(
    [pd.read_csv(f) for f in archivos], ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\nTotal fusionado: {len(dataset_completo)} filas')
print(f'Distribución de clases:\n{dataset_completo["Label"].value_counts().sort_index()}')

# Split 70 / 15 / 15
df_train, df_temp = train_test_split(dataset_completo, test_size=0.30, random_state=42)
df_val,   df_test = train_test_split(df_temp,          test_size=0.50, random_state=42)

df_train.to_csv('ml/data/processed/dataset/train/train.csv',         index=False)
df_val.to_csv('ml/data/processed/dataset/validation/validation.csv', index=False)
df_test.to_csv('ml/data/processed/dataset/test/test.csv',            index=False)

print(f'\nTrain: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}')

---
## 3. Entrena y exporta modelos
Entrena la red neuronal y exporta dos formatos:
- **`.keras`** — para reentrenar o inspeccionar en Python
- **`.tflite`** — para desplegar en el dispositivo Android

In [ ]:
# 1. Cargar los archivos generados en la Sección 2
df_train = pd.read_csv('ml/data/processed/dataset/train/train.csv')
df_val   = pd.read_csv('ml/data/processed/dataset/validation/validation.csv')
df_test  = pd.read_csv('ml/data/processed/dataset/test/test.csv')

# 2. Separar características (X) y etiquetas (y)
X_train = df_train.drop(columns=['Label']).values;  y_train = df_train['Label'].values
X_val   = df_val.drop(columns=['Label']).values;    y_val   = df_val['Label'].values
X_test  = df_test.drop(columns=['Label']).values;   y_test  = df_test['Label'].values

n_features = X_train.shape[1]
print(f'Variables de entrada: {n_features}')
print(f'Muestras totales: {len(X_train) + len(X_val) + len(X_test)}')

In [ ]:
# 2. Adaptar capa de normalización 
normalizer = keras.layers.Normalization(axis=-1)
normalizer.adapt(X_train) 

model_mlp = keras.Sequential([
    keras.layers.InputLayer(input_shape=(n_features,)),
    normalizer, 
    keras.layers.Dense(24, activation='relu'),
    keras.layers.Dense(12, activation='relu'),
    keras.layers.Dense(3,  activation='softmax')
], name='safedrive_mlp_v2')

model_mlp.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("Entrenando Red Neuronal...")
history = model_mlp.fit(X_train, y_train, epochs=40, batch_size=16, 
                        validation_data=(X_val, y_val), verbose=0)
print("¡Red Neuronal entrenada!")

In [ ]:
# 3. Entrenamiento — Random Forest (Baseline)
from sklearn.ensemble import RandomForestClassifier

print("Entrenando Random Forest para comparativa...")
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)
print("¡Random Forest entrenado!")

In [ ]:
# 4. Comparativa resultados
from sklearn.metrics import accuracy_score, classification_report

# Evaluación MLP
_, acc_mlp = model_mlp.evaluate(X_test, y_test, verbose=0)

# Evaluación Random Forest
preds_rf = model_rf.predict(X_test)
acc_rf = accuracy_score(y_test, preds_rf)

print("\n" + "="*40)
print("   RESULTADOS DE LA EXPERIMENTACIÓN")
print("="*40)
print(f"Accuracy MLP (Normalizado): {acc_mlp*100:.2f}%")
print(f"Accuracy Random Forest:     {acc_rf*100:.2f}%")
print("="*40)

# Informe detallado del modelo final (MLP)
print("\nInforme técnico del modelo MLP:")
preds_mlp = np.argmax(model_mlp.predict(X_test, verbose=0), axis=1)
print(classification_report(y_test, preds_mlp, target_names=['Normal', 'Susto', 'Choque']))

In [ ]:
# 6. Exportación (Keras y TFLite)
os.makedirs('models/v2', exist_ok=True)

# 1. Guardar .keras (Para ti, para el PC)
model_mlp.save('models/v2/safedrive_cu03_model.keras')

# 2. Guardar .tflite (Para el móvil, optimizado)
converter = tf.lite.TFLiteConverter.from_keras_model(model_mlp)
converter.optimizations = [tf.lite.Optimize.DEFAULT] # Cuantización para batería
tflite_model = converter.convert()

with open('models/v2/safedrive_cu03_model.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"\nModelos exportados correctamente a la carpeta 'models/v2/'")